# M1. 사전학습 모델 비교 — 같은 데이터·같은 설정에서 본체만 바꾸기

**질문**: 어떤 사전학습 모델이 우리 데이터(웹 뉴스, 7종 태그)에 가장 잘 맞는가?

**통제 실험**: 바꾸는 것은 `model` 하나뿐이다. 데이터(학습 30만 split_v3, dev·test는 split_v1과 동일), 최대 길이 512, 학습률 5e-5, 배치 16, 3에폭, 시드 42, 채점 함수(엄격 일치 span F1)는 모두 같다. 그래서 점수 차이를 모델 차이로 읽을 수 있다.

**팀 분담**: 아래 3번 셀의 `RUN` 에 자기 몫의 모델 이름만 넣고 돌린다. 결과는 같은 `reports/results` 폴더에 쌓이고, 이미 끝난 실험은 자동으로 건너뛴다. 마지막에 `R0_결과종합` 노트북이 한 표로 모은다.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 1. 후보 모델과 고른 이유

| 키 | 모델 | 구조 | 사전학습 데이터 | 이 비교에서 보려는 것 |
|---|---|---|---|---|
| `klue_roberta` | klue/roberta-base | RoBERTa, 12층 | 일반 한국어(뉴스·위키 등) | **기준선** |
| `klue_bert` | klue/bert-base | BERT, 12층 | 같은 KLUE 말뭉치 | RoBERTa와 BERT 학습 방식 차이 |
| `koelectra` | monologg/koelectra-base-v3-discriminator | ELECTRA, 12층 | 일반 한국어 | ELECTRA 구조 효과 (**대조군**) |
| `aihub_electra_1` | AI Hub 1세부 언어모델 | ELECTRA, 12층 | **이 말뭉치(뉴스 10억 어절)** | 같은 말뭉치로 사전학습한 효과 |
| `aihub_electra_2` | AI Hub 2세부 언어모델 | ELECTRA, 12층 | 이 말뭉치 | 1세부와 차이 |
| `klue_roberta_large` | klue/roberta-large | RoBERTa, 24층 | 일반 한국어 | 모델 크기 효과 (선택, 느림) |

**세 가지 사전학습 방식**
- **BERT**: 문장의 15% 단어를 가리고 맞히기(MLM) + 두 문장이 이어지는지 맞히기.
- **RoBERTa**: BERT와 같은 구조. 두 번째 과제를 빼고, 더 많은 데이터로 더 오래, 가리는 위치를 매번 바꿔(동적 마스킹) 학습.
- **ELECTRA**: 작은 생성기가 일부 단어를 그럴듯한 가짜로 바꾸고, 본체(판별기)가 **모든 토큰에 대해** "원래 단어인가, 바뀐 단어인가"를 맞힌다. 가려진 15%만이 아니라 모든 토큰에서 배우므로 같은 계산량에서 효율이 좋다. 토큰마다 판단하는 학습이라 토큰 분류(NER)와 성격이 비슷하다.

**왜 KoELECTRA를 대조군으로 넣나**: klue/roberta와 AI Hub ELECTRA만 비교하면 차이가 "RoBERTa냐 ELECTRA냐(구조)"와 "같은 말뭉치냐(데이터)" 두 원인으로 섞인다. 일반 한국어 ELECTRA를 함께 보면 `AI Hub ELECTRA − KoELECTRA` 가 순수하게 "같은 말뭉치 사전학습 효과"가 된다.

**형태소 입력 후보 (`_morph`)**: AI Hub 학습 코드를 보면 사전학습 때 형태소로 나눈 문장("실천해달라" → "실천 해 달 라")을 썼다. 원문을 그대로 넣으면 학습 때와 입력 모양이 다르다. 그래서 Kiwi 형태소 분석기로 형태소 경계에 공백만 끼워 넣은 입력도 후보로 둔다. 예) "중국인들의 언어습관에" → "중국인 들 의 언어 습관 에". 글자는 바꾸지 않고 위치 대응표로 예측을 원문 위치로 되돌리므로 **채점 기준은 다른 실험과 같다.**

| 실측 (test 1만 문장, 학습 없이) | 원문 입력 | 형태소 입력 |
|---|---|---|
| klue/roberta-base 점수 상한 F1 | 0.9706 | 0.9747 |
| AI Hub ELECTRA 1세부 점수 상한 F1 | 0.9670 | 0.9715 |
| 평균 토큰 수 (klue) | 46.7 | 46.0 |

개체의 74%가 조사 앞에서 끝나는데, 형태소로 나누면 개체 끝이 토큰 경계와 맞는 경우가 늘어 상한이 오른다. 실제 학습 점수가 오르는지는 이 실험으로 확인한다. 비교는 같은 모델끼리(`klue_roberta` vs `klue_roberta_morph`, `aihub_electra_1` vs `aihub_electra_1_morph`) 한다. 형태소 분석기가 AI Hub 구축 때 쓴 것과 같지 않을 수 있다는 점은 한계로 적는다.

In [ ]:
# AI Hub ELECTRA: 압축(zip 안의 zip)을 Colab 디스크에 풀고 폴더 경로를 얻는다 (1회, 1~2분)
AIHUB_ZIP = f"{PROJECT}/대규모 웹데이터 기반 한국어 말뭉치 데이터/AI모델 및 소스코드.zip"
AIHUB_1 = prepare_aihub_model(AIHUB_ZIP, "1세부")
AIHUB_2 = prepare_aihub_model(AIHUB_ZIP, "2세부")
print(AIHUB_1, os.listdir(AIHUB_1))

In [ ]:
ner_lib.DEFAULTS["stage"] = "M1_모델비교"
MODELS = {
    "klue_roberta":       dict(model="klue/roberta-base"),
    "klue_bert":          dict(model="klue/bert-base"),
    "koelectra":          dict(model="monologg/koelectra-base-v3-discriminator"),
    "aihub_electra_1":    dict(model=AIHUB_1),
    "aihub_electra_2":    dict(model=AIHUB_2),
    "klue_roberta_large": dict(model="klue/roberta-large", lr=2e-5, batch=8, grad_accum=2),  # 큰 모델: 배치 8×2로 같은 유효 배치 16
    # 형태소 입력: 문장을 형태소 경계에서 띄어 넣고(글자는 그대로), 예측은 원문 위치로 되돌려 같은 기준으로 채점
    "klue_roberta_morph":    dict(model="klue/roberta-base", pretok="morph"),
    "aihub_electra_1_morph": dict(model=AIHUB_1, pretok="morph"),
}
COMMON = dict(train=TRAIN, dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl",
              out_dir=RESULTS, max_len=512, lr=5e-5, epochs=3, batch=16, seed=42)

## 2. 토크나이저 비교 (학습 없이 1~2분)
모델마다 토크나이저가 다르다. 같은 문장을 몇 개 토큰으로 자르는지, 정답을 BIO로 바꿨다 되돌렸을 때 점수 상한이 얼마인지 먼저 본다. 토큰이 적을수록 빠르고, 상한이 높을수록 경계를 정확히 표현할 수 있다.

In [ ]:
from transformers import AutoTokenizer
test_rows = load_jsonl(f"{SPLIT}/test.jsonl")
for key, m in MODELS.items():
    tok = AutoTokenizer.from_pretrained(m["model"]); pt = m.get("pretok")
    n_tok = sum(len(ner_lib.tokenize_offsets(r["text"], tok, 512, pt)["input_ids"]) for r in test_rows[:2000]) / 2000
    print(f"{key:22s} 평균 토큰 {n_tok:5.1f} | 어휘 {tok.vocab_size:6d} | test 상한 F1 {upper_bound(test_rows, tok, 512, pt)['f1']}")

## 3. 학습 실행
`RUN` 에 돌릴 모델 키를 넣는다. 처음이면 `klue_roberta` 하나로 시간부터 잰다. 끝나면 `학습 n s (k 문장/s)` 가 찍히는데, 이 속도로 나머지 모델 시간을 예측할 수 있다(large는 약 3배 느림).

Colab 연결이 끊겨도 끝난 모델은 결과 json이 남아 있어 다시 돌리면 건너뛴다.

In [ ]:
RUN = ["klue_roberta", "koelectra", "aihub_electra_1"]     # ← 자기 몫으로 바꾸기 (형태소 입력: "klue_roberta_morph", "aihub_electra_1_morph")
LRS = [None]          # 조정할 때: [None, 3e-5] 처럼. None = 모델 기본값(5e-5, large는 2e-5). 모든 모델에 같은 개수로!
for key in RUN:
    for lr in LRS:
        cfg = {**COMMON, **MODELS[key], "exp": f"M1_{key}{SUFFIX}" + (f"_lr{lr:g}" if lr else "")}
        if lr: cfg["lr"] = lr
        run_experiment(cfg)

## 4. 에폭 수 정하기 — 에폭별 dev 점수
학습 중 매 에폭 끝에 dev 점수를 쟀다. 선이 어디서 평평해지거나 꺾이는지 보고 에폭 수를 정한다. **dev 점수로만 정하고 test는 보지 않는다.**

**조정 공정성 규칙**: 학습률·에폭 같은 설정 조정은 모든 모델에 **같은 횟수**만 한다. 한 모델만 여러 번 조정하면 모델이 좋아서가 아니라 조정을 많이 해서 이긴 것이 된다.

In [ ]:
import matplotlib.pyplot as plt
for r in load_results(RESULTS):
    if r["exp"].startswith("M1_") and r.get("history"):
        h = r["history"]; plt.plot([x["epoch"] for x in h], [x["dev_f1"] for x in h], "o-", label=r["exp"])
plt.xlabel("epoch"); plt.ylabel("dev F1"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

## 5. 결과 보기
- `test_F1`: 전체 개체 단위 F1 (micro). 개체가 많은 CV·OG의 영향이 크다.
- `macro_F1`: 7개 태그 F1의 평균. 드문 태그(NW·TR·FD)를 얼마나 잡는지가 더 드러난다.
- 차이가 작으면(예: 0.5점 이하) 우연일 수 있다. `D2_표본흔들림` 노트북에서 잰 흔들림 크기와 비교해서 판단한다.

In [ ]:
df = results_table([r for r in load_results(RESULTS) if r["exp"].startswith("M1_")])
df.sort_values("test_F1", ascending=False)

In [ ]:
# 2단계(금융) 사전 지표: 경제·산업 기사에서의 F1
for r in load_results(RESULTS):
    if r["exp"].startswith("M1_"):
        cat = r["test_by_category"]
        print(f"{r['exp']:24s} 경제 {cat.get('경제',{}).get('f1')} | 산업 {cat.get('산업',{}).get('f1')}")

## 가중치 저장 확인
이번에 돌린 실험마다 `내 드라이브/ner_models/{실험}/` 에 가중치가 있는지 확인한다. 없으면 에러로 멈춘다.

In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
